# QC0426 / VTI — Measurement (round 2)

**QC0426** (trilayer graphene, stacking under investigation) on the
**VTI** rig, 2026-06-01 cooldown. Round-2 HERA live session: the operator
runs every cell; the agent is advisory via nb-collab.

`metadata.hera.policy = append_only` — scientific record: append new cells;
don't edit or delete executed ones.

Confirm each `# <-- verify` line at bring-up. Cells marked **PATTERN** /
**PROPOSAL** contain `<PLACEHOLDER>` values or pending decisions — review
before running.

## Setup

In [1]:
import os
import time

import numpy as np
import scipy.constants as cst

import qcodes
from qcodes import validators
from qcodes.instrument_drivers.stanford_research import SR830
from qcodes.instrument_drivers.tektronix.Keithley_2400 import Keithley_2400

import sweep

from drivers.FastBi1K5 import FastBi1K5
from drivers.Lakeshore_335 import Lakeshore_335
from drivers.SRS_DC205 import SRS_DC205
from drivers.async_station import async_station

In [2]:
main_dir = os.getcwd()
sweep.set_basedir(os.path.join(main_dir, "data"))
main_dir

'C:\\Users\\DGG\\VTI_local_data\\zack\\20260614-QC0426'

# Connect Instruments

## Magnet

In [3]:
magnet = FastBi1K5('magnet', '192.168.0.10')  # <-- verify address
magnet.status()

Connected to: Fast-Bi-1K5 Magnet Controller
---------- Magnet Status ----------
Field:               : 0.002 Tesla
Current:             : 0.017 A
Voltage:             : -0.000 V
Output Enabled       : True
Persistent Switch On : True
Loop Mode            : CC
Update Mode          : NORMAL
-----------------------------------


## Lakeshore

In [4]:
heater = Lakeshore_335('l335_3', 'GPIB0::10::INSTR')  # <-- verify address
print(heater.sample_temperature())
print(heater.vaporizer_temperature())

Connected to: LSCI 335 (serial:LSA2FM7/#######, firmware:1.4) in 0.40s
1.163
0.0


## Lock-ins

In [5]:
sr0 = SR830('sr0', "GPIB0::17::INSTR")  # <-- verify — current channel (Ithaco preamp output)
sr1 = SR830('sr1', "GPIB0::14::INSTR")  # <-- verify — also the AC drive source
sr2 = SR830('sr2', "GPIB0::19::INSTR")  # <-- verify
sr3 = SR830('sr3', "GPIB0::18::INSTR")  # <-- verify

C:\Users\DGG\measurement_env\.venv\Lib\site-packages\qcodes\instrument\instrument_base.py:188: QCoDeSDeprecationWarning: Parameter ch1_databuffer on instrument sr0 does not correctly pass kwargs to its baseclass. A Parameter class must take `**kwargs` and forward them to its baseclass.
  warnings.warn(
C:\Users\DGG\measurement_env\.venv\Lib\site-packages\qcodes\instrument\instrument_base.py:188: QCoDeSDeprecationWarning: Parameter ch2_databuffer on instrument sr0 does not correctly pass kwargs to its baseclass. A Parameter class must take `**kwargs` and forward them to its baseclass.
  warnings.warn(
C:\Users\DGG\measurement_env\.venv\Lib\site-packages\qcodes\instrument\instrument_base.py:188: QCoDeSDeprecationWarning: Parameter ch1_databuffer on instrument sr1 does not correctly pass kwargs to its baseclass. A Parameter class must take `**kwargs` and forward them to its baseclass.
  warnings.warn(
C:\Users\DGG\measurement_env\.venv\Lib\site-packages\qcodes\instrument\instrument_base.p

Connected to: Stanford_Research_Systems SR830 (serial:s/n45620, firmware:ver1.07) in 0.08s
Connected to: Stanford_Research_Systems SR830 (serial:s/n83412, firmware:ver1.07) in 0.08s
Connected to: Stanford_Research_Systems SR830 (serial:s/n41639, firmware:ver1.07) in 0.09s
Connected to: Stanford_Research_Systems SR830 (serial:s/n93552, firmware:ver1.07) in 0.09s


C:\Users\DGG\measurement_env\.venv\Lib\site-packages\qcodes\instrument\instrument_base.py:188: QCoDeSDeprecationWarning: Parameter ch1_databuffer on instrument sr3 does not correctly pass kwargs to its baseclass. A Parameter class must take `**kwargs` and forward them to its baseclass.
  warnings.warn(
C:\Users\DGG\measurement_env\.venv\Lib\site-packages\qcodes\instrument\instrument_base.py:188: QCoDeSDeprecationWarning: Parameter ch2_databuffer on instrument sr3 does not correctly pass kwargs to its baseclass. A Parameter class must take `**kwargs` and forward them to its baseclass.
  warnings.warn(


## Keithleys

In [6]:
k_bg = Keithley_2400('k_bg', 'GPIB0::24::INSTR')  # <-- verify address — back gate
k_bg.mode.set('VOLT')
k_bg.volt.set(0)
k_bg.rangei.set(1e-6)
k_bg.rangev.set(21)
k_bg.volt.step = 0.005

Connected to: KEITHLEY INSTRUMENTS INC. 2400 (serial:0734554, firmware:C17   Jul 16 1999 15:20:40/A02  /G/F) in 0.03s


In [7]:
k_2410 = Keithley_2400('k_2410', 'GPIB0::25::INSTR')  # <-- verify address — Si global gate
k_2410.mode.set('VOLT')
k_2410.volt.set(0)
k_2410.rangei.set(1e-6)
k_2410.rangev.set(21)
k_2410.volt.step = 0.005

Connected to: KEITHLEY INSTRUMENTS INC. 2410 (serial:1293333, firmware:C30   Mar 17 2006 09:29:29/A02  /J/J) in 0.03s


## DC205 (top gate)

In [8]:
k_tg = SRS_DC205("k_tg", "ASRL7::INSTR")  # <-- verify address — top gate
k_tg.ramp_rate = 2  # V/s

Connected to: Stanford_Research_Systems DC205 (serial:s/n20500514, firmware:ver1.80) in 0.03s


In [9]:
# Sweepable top-gate parameter — appears in the data as column 'DC_bias'.
def dcbias(vg):
    k_tg.ramp_voltage(vg)

dc205_volt = qcodes.Parameter('DC_bias', unit='V', set_cmd=dcbias)

# SAFETIES

Run immediately after instrument bring-up, before any sweep.

In [10]:
# Gate validators — QC0426 GATE OPERATING LIMITS (see device card).
# Provided operating limits, not range-found this session (range-finding
# is blocklisted — see the brief). Commit exactly the device-card numbers.
TG_MIN, TG_MAX = -6.0, +7.0   # top gate (dc205_volt / k_tg), V — provided
BG_MIN, BG_MAX = -4.5, +4.5   # back gate (k_bg), V — provided
SI_MIN, SI_MAX = -50.0, +50.0  # Si gate (k_2410), V — provided; operating point per-run
dc205_volt.vals = validators.Numbers(TG_MIN, TG_MAX)
# Instrument-level TG limit: direct k_tg.voltage()/ramp_voltage() calls
# bypass the dc205_volt wrapper; ramp steps also go through k_tg.voltage.
k_tg.voltage.vals = validators.Numbers(TG_MIN, TG_MAX)
k_bg.volt.vals = validators.Numbers(BG_MIN, BG_MAX)
k_2410.volt.vals = validators.Numbers(SI_MIN, SI_MAX)

# Limit lists for sanitize_tg_bg()
tg_lims, bg_lims = [TG_MIN, TG_MAX], [BG_MIN, BG_MAX]

# Session field cap — machine-enforced: magnet.mag is a qcodes Parameter,
# so this validator rejects any magnet.mag(target) with |B| > B_SESSION_CAP
# before any hardware traffic. (Driver max_field_T = 14.0 remains as backstop.)
B_SESSION_CAP = 8.0  # Tesla
magnet.mag.vals = validators.Numbers(-B_SESSION_CAP, B_SESSION_CAP)

# Temperature setpoint cap — TYPO GUARD ONLY (sample_setpoint(3000) vs 300).
# The real safe T/B envelope is a rig-card item, unestablished this session.
T_SESSION_CAP = 310.0  # K  # <-- verify
heater.sample_setpoint.vals = validators.Numbers(0, T_SESSION_CAP)
heater.vaporizer_setpoint.vals = validators.Numbers(0, T_SESSION_CAP)


# Helpers

In [11]:
def spoints(a, b, c):
    return np.linspace(a, b, c)


def sanitize_tg_bg(vtg_vec, vbg_vec, tg_lims, bg_lims):
    """
    Given a list of tg and bg voltages, removes any points which are outside tg_lims and bg_lims
    """
    vtg_new_vec = []
    vbg_new_vec = []
    for ind in range(len(vtg_vec)):
        vtg, vbg = vtg_vec[ind], vbg_vec[ind]
        if vtg < tg_lims[0] or vtg > tg_lims[1] or vbg < bg_lims[0] or vbg > bg_lims[1]:
            pass
        else:
            vtg_new_vec.append(vtg)
            vbg_new_vec.append(vbg)
    return np.array(vtg_new_vec), np.array(vbg_new_vec)

In [12]:
cst_epsilon0 = cst.epsilon_0
cst_e = cst.e
cst_h = cst.h
cst_phi0 = cst_h/(2*cst_e)
cst_kappa_SiO2 = 3.9
cst_kappa_hBN = 3

def calc_ctg(t_tg):
    """
    Calculates area capacitance for a top gate
    """
    return cst_epsilon0*cst_kappa_hBN/t_tg

def calc_cbg(t_bg, t_bgSiO2):
    """
    Calculates area capacitance for a back gate
    For a metal back gate, set t_bgSiO2 = 0
    """
    return cst_epsilon0*1/(t_bg/cst_kappa_hBN + t_bgSiO2/cst_kappa_SiO2)

def calc_nD(V_tg, V_bg, c_tg, c_bg, n_offset=0, D_offset = 0):
    n = (1/cst_e) * (c_tg*V_tg  + c_bg*V_bg) + n_offset
    D = 0.5*(c_tg*V_tg - c_bg*V_bg)*1/cst_epsilon0 + D_offset
    return n, D


# QC0426 gate stack geometry (fabrication facts; GEOMETRY PRIOR ONLY —
# calibrate via cnp-and-dline-calibration before trusting derived scales).
# SiO2 term zeroed — (n, D) uses the hBN top + back gates only; the Si
# gate (k_2410) is a screening gate, set per-run.
t_tg = 15.6e-9  # m
t_bg = 12e-9    # m
t_bgSiO2 = 0    # m — zeroed for QC0426

c_tg = calc_ctg(t_tg)
c_bg = calc_cbg(t_bg, t_bgSiO2)

# Measurement config

In [13]:
# QC0426 measurement config — SKELETON. Contact pairs are per-run: set
# them per run (or let ContactMux write the mc_fragment once the QSwitch
# is live). Always set 'comments' per run — structured, current, no
# inherited text.
meas_config = {
    "sr0": "drain current on <CONTACT>",  # <-- set per wiring
    "sr0_gain": -1e6,  # ithaco gain
    "sr1": "<PAIR>",
    "sr2": "<PAIR>",
    "sr3": "<PAIR>",
    "bias_resistor": 100e6,
    "bias_voltage": sr1.amplitude(),
    "lockin_freq": sr1.frequency(),
    "sr0_sens": sr0.sensitivity(),
    "sr1_sens": sr1.sensitivity(),
    "sr2_sens": sr2.sensitivity(),
    "sr3_sens": sr3.sensitivity(),
    "sample_temperature:": heater.sample_temperature(),  # key verbatim (trailing ':') to match this cooldown's metadata
    "Si (V)": k_2410.volt(),
    "comments": "<SET PER RUN>",
}

## QSwitch (contact mux) — LIVE (validated 2026-06-14)

Bring-up complete (`qswitch-smoke-test.md` step 7): QSwitch on **ASRL4**,
Si standing closures on lines **16/11** confirmed closed, `settle_s` =
**4 s**. Maps below are the device card's, as deployed. `qsw.reset()` is
**forbidden once Si is energized** — it opens the standing closures and
drops the Si gate. contact-survey-2t's automated flavor is available.

In [14]:
from drivers.QSwitch import QSwitch
from drivers.contact_mux import ContactMux

qsw = QSwitch('qswitch', 'ASRL4::INSTR')   # validated 2026-06-14

# Contact label -> QSwitch line (= F1 fischer index). Si lines 16/11 are
# EXCLUDED (standing closures, not managed); TG/BG are hardwired off-switch.
LINES = {
    "T1": 1,  "T2": 3,  "T3": 4,  "T4": 6,  "T5": 7,  "T6": 12, "T7": 13,
    "B1": 8,  "B2": 17, "B3": 23, "B4": 22, "B5": 20, "B6": 19, "B7": 18, "B8": 15,
}
# Instrument port -> front-panel BNC tap.
BREAKOUTS = {
    "source": 7, "drain": 8,
    "sr1+": 1, "sr1-": 2, "sr2+": 3, "sr2-": 4, "sr3+": 5, "sr3-": 6,
}
SETTLE_S = 4   # calibrated 2026-06-14

mux = ContactMux(qsw, lines=LINES, breakouts=BREAKOUTS, settle_s=SETTLE_S)
# qsw.reset() FORBIDDEN once Si is energized — drops the Si gate (lines 16/11).

Connected to: Quantum Machines QSwitch (serial:37, firmware:2.0) in 0.06s


In [23]:
print('before', qsw.state())
print(k_2410.volt())

qsw.close_relay(11, 9)
qsw.close_relay(16, 9)

qsw.open_relay(11, 0)
qsw.open_relay(16, 0)

print('after', qsw.state())

before (@1!0:24!0,11!9,16!9)
0.0
after (@1!0:10!0,12!0:15!0,17!0:24!0,11!9,16!9)


## HERA — contact-survey-2t (first-session 2T screen) — run 0: T1–T7

**Recipe**: `measurements/contact-survey-2t` (mux/automated flavor, first-session branch). No calibrated D-line exists yet — that is recipe 2's product — so this screens each pair with a **TG-only sweep at BG = Si = 0**: a pair that conducts through the gated channel shows a CNP **dip** in the 2T current (sr0). TG −6→+7 V, **53 pts (0.25 V/pt)**, delay 0.5 s. Sensitivities are **fixed, not autoranged** — comparability across pairs is the survey's product.

**Why pair 1 alone**: the recipe's Verify mandates reading run-0 `metadata.json` (qswitch_state/routing match intent; `sr0`/pair keys present) **before** looping the set, and run 0 also calibrates the realized per-point time I'll price the rest on. After it lands I'll verify truth + timing + whether the gated dip is present, then land batch 2 (pairs 2–4 path-finding, then a qualifying star).

**Coverage trade**: a single density line (TG at BG=Si=0), not a gate map. If *no* pair shows a CNP dip, the CNP is off this line and I'll add a BG offset and re-screen — stated, cheap to extend.

**Eyeball before running**: preflight asserts T at base (1.0–1.5 K) and B ≈ 0; drives BG→0, Si→0; prints sr0 sensitivity. Si standing closures (lines 16/11) are **not** in the mux managed set, so `select()` never touches them. **Duration** ~75 s (pair 1); full path-finding batch (4 pairs) ~5 min to follow.

In [24]:
import numpy as np

# ---- Frozen survey conditions (step 1: identical for every pair) ----
k_bg.volt(0.0)        # back gate = 0 V (frozen)
k_2410.volt(0.0)      # Si = 0 V (frozen; already at 0)
TG_VEC = np.linspace(-6.0, 7.0, 53)   # TG screen, 0.25 V/pt, inside validators
SURVEY_DELAY = 0.5    # s/pt dwell; run 0 calibrates realized per-point

# ---- Preflight: silent preconditions this sweep assumes but does not set ----
T = heater.sample_temperature(); assert 1.0 < T < 1.5, f'T off base: {T}'
B = magnet.mag();                assert abs(B) < 0.05, f'B not ~0: {B}'
print(f'preflight: T={T:.3f} K, B={B:.4f} T, BG={k_bg.volt():.3f} V, Si={k_2410.volt():.2f} V')
print(f'sr0 sens={sr0.sensitivity()}  (current chan; fixed, NOT autoranged)')

def survey_pair(label, src, drn):
    mux.select({'source': src, 'drain': drn})   # make-before-break; Si 16/11 untouched
    mc = {**meas_config,
          'sr1': 'n/a (2T survey)', 'sr2': 'n/a (2T survey)', 'sr3': 'n/a (2T survey)',
          'comments': (f'contact-survey-2t | 2T pair {label} src={src} drn={drn} | '
                       f'TG {TG_VEC[0]:.1f}..{TG_VEC[-1]:.1f} V x{len(TG_VEC)} | '
                       f'BG=0 Si=0 B=0 T~{T:.2f}K')}
    mc = mc | mux.mc_fragment()                 # per-run contact truth
    s = async_station(mc)
    s.fp(sr0.X).fp(sr0.Y)
    s.add_comment('contact-survey-2t')
    s.sweep(dc205_volt, TG_VEC, delay=SURVEY_DELAY)

# ---- Run pair 1 only: path-finder + run-0 truth/timing calibration ----
survey_pair('T1-T7', 'T1', 'T7')

preflight: T=1.163 K, B=0.0023 T, BG=0.000 V, Si=0.00 V
sr0 sens=0.01  (current chan; fixed, NOT autoranged)
2026-06-14 21:02:37,390 [INFO] Starting sweep with ID 0
2026-06-14 21:02:37,391 [INFO] Minimum duration 0h 0m 26s


2026-06-14 21:03:16,685 [INFO] Completed in 0h 0m 39s
2026-06-14 21:03:16,685 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\0\data.tsv.gz


## HERA — contact-survey-2t batch 2: path-finding mini-star on T4

**Run 0 (T1–T7) read dead/high-R**: ~0.34 nA flat vs 5 nA nominal, gate-independent (~1.3 GΩ) — see `data/0`. Either T1 or T7 (or both) is open/high-R, or no gated channel spans that route. Truth + timing verified (routing correct, ~0.74 s/pt), so I'm now looping the rest.

**Design**: a mini-star centered on **T4** (a central top contact), 6 pairs. T4 appears in 5 → if T4 is good, every partner is ranked against it in one pass; if T4 itself is the bad anchor, all 5 read dead and the 6th pair (**T3–T5**, no T4) is the hedge that tells me whether *any* top contacts conduct. Adjacent pairs (T4–T5, T3–T4) come first — shortest channel, most likely to light up clean. T1–T4 / T4–T7 re-test the pair-0 suspects. T4–B4 probes a transverse path.

**Conditions identical to run 0** (frozen): BG=Si=0, TG −6→+7 V ×53, delay 0.5 s, fixed sensitivities — comparability is the product. Re-asserts T at base and B≈0 up front. **Duration** ~4 min (6 × ~40 s). Writes `data/1`–`data/6`. After it lands I rank the contacts and either choose a workable S/D + probe candidates or pivot the anchor.

In [25]:
# Batch 2 — path-finding mini-star on T4 (+ T3-T5 hedge). survey_pair() defined in cell 29.
assert 1.0 < heater.sample_temperature() < 1.5, 'T off base'
assert abs(magnet.mag()) < 0.05, 'B not ~0'
for label, src, drn in [
    ('T4-T5', 'T4', 'T5'),   # adjacent central top: shortest channel, most likely clean
    ('T3-T4', 'T3', 'T4'),
    ('T1-T4', 'T1', 'T4'),   # re-tests T1 (pair-0 suspect)
    ('T4-T7', 'T4', 'T7'),   # re-tests T7 (pair-0 suspect)
    ('T4-B4', 'T4', 'B4'),   # transverse path
    ('T3-T5', 'T3', 'T5'),   # hedge: top pair with NO T4, in case T4 is the bad anchor
]:
    survey_pair(label, src, drn)
    print('done', label)

2026-06-14 21:08:22,083 [INFO] Starting sweep with ID 1
2026-06-14 21:08:22,093 [INFO] Minimum duration 0h 0m 26s


2026-06-14 21:09:04,984 [INFO] Completed in 0h 0m 42s
2026-06-14 21:09:04,984 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\1\data.tsv.gz
done T4-T5
2026-06-14 21:09:09,504 [INFO] Starting sweep with ID 2
2026-06-14 21:09:09,505 [INFO] Minimum duration 0h 0m 26s


2026-06-14 21:09:52,342 [INFO] Completed in 0h 0m 42s
2026-06-14 21:09:52,343 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\2\data.tsv.gz
done T3-T4
2026-06-14 21:09:56,865 [INFO] Starting sweep with ID 3
2026-06-14 21:09:56,866 [INFO] Minimum duration 0h 0m 26s


2026-06-14 21:10:39,682 [INFO] Completed in 0h 0m 42s
2026-06-14 21:10:39,683 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\3\data.tsv.gz
done T1-T4
2026-06-14 21:10:44,205 [INFO] Starting sweep with ID 4
2026-06-14 21:10:44,206 [INFO] Minimum duration 0h 0m 26s


2026-06-14 21:11:27,042 [INFO] Completed in 0h 0m 42s
2026-06-14 21:11:27,042 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\4\data.tsv.gz
done T4-T7
2026-06-14 21:11:31,565 [INFO] Starting sweep with ID 5
2026-06-14 21:11:31,566 [INFO] Minimum duration 0h 0m 26s


2026-06-14 21:12:14,462 [INFO] Completed in 0h 0m 42s
2026-06-14 21:12:14,462 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\5\data.tsv.gz
done T4-B4
2026-06-14 21:12:18,982 [INFO] Starting sweep with ID 6
2026-06-14 21:12:18,982 [INFO] Minimum duration 0h 0m 26s


2026-06-14 21:13:01,834 [INFO] Completed in 0h 0m 42s
2026-06-14 21:13:01,835 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\6\data.tsv.gz
done T3-T5


## HERA — contact-survey-2t batch 3: complete the T4-star (9 remaining contacts)

**Batch 2 result**: **T1 localized DEAD** (high-R in both T1–T7 #0 and T1–T4 #3 while partners T4/T7 conduct ~5 nA elsewhere). **T3, T4, T5, T7, B4 alive.** T4 is now a confirmed-good anchor.

**This batch** runs T4 against the 9 still-untested contacts (T2, T6, B1, B2, B3, B5, B6, B7, B8) — a single clean star off a known-good anchor maps every remaining contact in one ~6 min pass (any dead partner localizes to itself, since T4 is good). After it I'll have the full alive/dead map and bless a **workable** S/D + three probe pairs with top+bottom region coverage (satisfice, not optimize), then move to recipe 2.

**Note**: the 2T *current* saturates at ~5 nA for every alive pair (current-bias through 100 MΩ), so this ranks aliveness, not resistance — the CNP and 4T resistances come from recipe 2's gate map (V = I·R). Conditions identical (BG=Si=0, TG −6→+7 ×53, 0.5 s). Writes `data/7`–`data/15`. ~6 min.

In [26]:
# Batch 3 — complete the T4-star over all remaining untested contacts.
assert 1.0 < heater.sample_temperature() < 1.5, 'T off base'
assert abs(magnet.mag()) < 0.05, 'B not ~0'
for label, src, drn in [
    ('T4-T2','T4','T2'), ('T4-T6','T4','T6'),
    ('T4-B1','T4','B1'), ('T4-B2','T4','B2'), ('T4-B3','T4','B3'),
    ('T4-B5','T4','B5'), ('T4-B6','T4','B6'), ('T4-B7','T4','B7'), ('T4-B8','T4','B8'),
]:
    survey_pair(label, src, drn)
    print('done', label)

2026-06-14 21:17:51,827 [INFO] Starting sweep with ID 7
2026-06-14 21:17:51,828 [INFO] Minimum duration 0h 0m 26s


2026-06-14 21:18:34,647 [INFO] Completed in 0h 0m 42s
2026-06-14 21:18:34,648 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\7\data.tsv.gz
done T4-T2
2026-06-14 21:18:39,171 [INFO] Starting sweep with ID 8
2026-06-14 21:18:39,172 [INFO] Minimum duration 0h 0m 26s


2026-06-14 21:19:22,022 [INFO] Completed in 0h 0m 42s
2026-06-14 21:19:22,023 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\8\data.tsv.gz
done T4-T6
2026-06-14 21:19:26,544 [INFO] Starting sweep with ID 9
2026-06-14 21:19:26,544 [INFO] Minimum duration 0h 0m 26s


2026-06-14 21:20:09,381 [INFO] Completed in 0h 0m 42s
2026-06-14 21:20:09,381 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\9\data.tsv.gz
done T4-B1
2026-06-14 21:20:13,903 [INFO] Starting sweep with ID 10
2026-06-14 21:20:13,903 [INFO] Minimum duration 0h 0m 26s


2026-06-14 21:20:56,765 [INFO] Completed in 0h 0m 42s
2026-06-14 21:20:56,766 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\10\data.tsv.gz
done T4-B2
2026-06-14 21:21:01,290 [INFO] Starting sweep with ID 11
2026-06-14 21:21:01,290 [INFO] Minimum duration 0h 0m 26s


2026-06-14 21:21:44,100 [INFO] Completed in 0h 0m 42s
2026-06-14 21:21:44,101 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\11\data.tsv.gz
done T4-B3
2026-06-14 21:21:48,624 [INFO] Starting sweep with ID 12
2026-06-14 21:21:48,624 [INFO] Minimum duration 0h 0m 26s


2026-06-14 21:22:31,443 [INFO] Completed in 0h 0m 42s
2026-06-14 21:22:31,444 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\12\data.tsv.gz
done T4-B5
2026-06-14 21:22:35,969 [INFO] Starting sweep with ID 13
2026-06-14 21:22:35,969 [INFO] Minimum duration 0h 0m 26s


2026-06-14 21:23:18,822 [INFO] Completed in 0h 0m 42s
2026-06-14 21:23:18,822 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\13\data.tsv.gz
done T4-B6
2026-06-14 21:23:23,347 [INFO] Starting sweep with ID 14
2026-06-14 21:23:23,348 [INFO] Minimum duration 0h 0m 26s


2026-06-14 21:24:06,220 [INFO] Completed in 0h 0m 42s
2026-06-14 21:24:06,220 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\14\data.tsv.gz
done T4-B7
2026-06-14 21:24:10,745 [INFO] Starting sweep with ID 15
2026-06-14 21:24:10,745 [INFO] Minimum duration 0h 0m 26s


2026-06-14 21:24:53,623 [INFO] Completed in 0h 0m 42s
2026-06-14 21:24:53,624 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\15\data.tsv.gz
done T4-B8


## HERA — 4T validation cut on blessed config (T2/T7 + T3–T4, B4–B5, T5–B6)

**Survey complete** (file_id 0–15): alive = T2,T3,T4,T5,T7 (top), B3,B4,B5,B6,B7 (bottom); dead = T1,T6,B1,B2,B8 (T1 localized via two bad pairs against good partners; the rest dead against good anchor T4).

**Blessed config** (8 distinct contacts, each routes to one tap): **source T2 / drain T7**; **sr1 = T3–T4** (Rxx top), **sr2 = B4–B5** (Rxx bottom), **sr3 = T5–B6** (Rxy). Two opposite-edge Rxx for region coverage + one Hall pair.

**This cut** routes the full 4T config and sweeps TG −6→+7 V (131 pts, 0.1 V/pt, delay 0.5 s) at BG=Si=0, reading all three 4T voltages + drain current. Purpose: confirm the probes give sensible, non-railed voltages and scout where the CNP sits relative to the BG=0 line — **before** committing to recipe 2's 2D gate map. Preflight prints sr1/sr2/sr3 sensitivities (watch for overload → flat-topped data; I'll re-range if so). **Duration** ~1.5 min. Writes `data/16`. If probes look good I write the config to the device card and design the gate map; if a probe rails or reads dead, I re-range or swap it.

In [27]:
# 4T validation cut on the blessed configuration.
assert 1.0 < heater.sample_temperature() < 1.5, 'T off base'
assert abs(magnet.mag()) < 0.05, 'B not ~0'
print('sens sr1/sr2/sr3 =', sr1.sensitivity(), sr2.sensitivity(), sr3.sensitivity())

ROUTING_4T = {'source':'T2','drain':'T7',
              'sr1+':'T3','sr1-':'T4',   # Rxx top
              'sr2+':'B4','sr2-':'B5',   # Rxx bottom
              'sr3+':'T5','sr3-':'B6'}   # Rxy transverse
mux.select(ROUTING_4T)                   # Si 16/11 untouched (not managed)

mc = {**meas_config,
      'comments': 'contact-survey-2t step5 | 4T-confirm blessed cfg src=T2 drn=T7 '
                  'sr1=T3-T4(Rxx) sr2=B4-B5(Rxx) sr3=T5-B6(Rxy) | TG -6..7 x131 BG=0 Si=0 B=0'}
mc = mc | mux.mc_fragment()
s = async_station(mc)
s.fp(sr0.X).fp(sr1.X).fp(sr2.X).fp(sr3.X).fp(sr0.Y)
s.add_comment('contact-survey-2t')
s.sweep(dc205_volt, np.linspace(-6.0, 7.0, 131), delay=0.5)

sens sr1/sr2/sr3 = 0.0005 0.0005 0.0002
2026-06-14 21:33:46,673 [INFO] Starting sweep with ID 16
2026-06-14 21:33:46,673 [INFO] Minimum duration 0h 1m 5s


2026-06-14 21:35:15,902 [INFO] Completed in 0h 1m 29s
2026-06-14 21:35:15,903 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\16\data.tsv.gz


## HERA — recipe 2: 0 T TG–BG gate map (CNP/D-line calibration source)

**Recipe**: `device-init/cnp-and-dline-calibration` (its input gate map) on the blessed config. 2D megasweep: **slow = BG [−4.5,+4.5] V × 21 rows** (0.45 V/row), **fast = TG [−6,+7] V × 53** (0.25 V/pt). Reads the three 4T voltages + drain current. The scout put the CNP at Vtg ≈ 0 (BG=0); with the geometry-prior slope (−c_bg/c_tg ≈ −1.3) the CNP ridge sweeps Vtg ≈ +5.9 → −5.9 across the BG rows — inside the TG window at every row, so a full-box rectangular map captures it without re-mapping.

**After**: re-form per channel from the TSV (`plot.png` is not the physical map), argmax the Rxx ridge per BG row, `polyfit` slope → **c_bg/c_tg = −slope** (ratio: data-calibrated; absolute scale stays geometry-prior until a QH lever-arm check), density offset from the intercept, then design constant-D lines clipped to validators. Also validates sr2/sr3 across the plane. **Point density**: 0.25 V/pt resolves the ~1 V CNP peak; 21 rows for the slope fit.

**Price** ~17 min (21 rows × [53 × ~0.74 s/pt + ~6.5 s TG reset + 2 s BG settle]), calibrated from the 0.74 s/pt scout (file_id 0). <30 min so no mid-run checkpoint; I'll read the full map after. Writes `data/17`. Preflight asserts T base, B≈0; re-selects the blessed routing (idempotent).

In [28]:
# Recipe 2 source gate map: 0 T TG-BG megasweep on the blessed 4T config.
assert 1.0 < heater.sample_temperature() < 1.5, 'T off base'
assert abs(magnet.mag()) < 0.05, 'B not ~0'
assert abs(k_2410.volt()) < 0.01, 'Si not 0'

ROUTING_4T = {'source':'T2','drain':'T7',
              'sr1+':'T3','sr1-':'T4', 'sr2+':'B4','sr2-':'B5', 'sr3+':'T5','sr3-':'B6'}
mux.select(ROUTING_4T)                 # idempotent; Si 16/11 untouched

BG_VEC = np.linspace(-4.5, 4.5, 21)    # slow, 0.45 V/row
TG_VEC = np.linspace(-6.0, 7.0, 53)    # fast, 0.25 V/pt

mc = {**meas_config,
      'comments': 'cnp-and-dline-calibration | 0T TG-BG gatemap | blessed cfg '
                  'src=T2 drn=T7 sr1=T3-T4 sr2=B4-B5 sr3=T5-B6 | '
                  'BG[-4.5,4.5]x21 TG[-6,7]x53 Si=0'}
mc = mc | mux.mc_fragment()
s = async_station(mc)
s.fp(sr0.X).fp(sr1.X).fp(sr2.X).fp(sr3.X).fp(sr0.Y)
s.add_comment('cnp-and-dline-calibration')
s.megasweep(k_bg.volt, BG_VEC, dc205_volt, TG_VEC, slow_delay=2.0, fast_delay=0.5)

2026-06-14 21:45:33,343 [INFO] Starting megasweep with ID 17
2026-06-14 21:45:33,344 [INFO] Minimum duration 0h 9m 58s


2026-06-14 21:53:04,340 [WARNING] ID 17 INTERRUPTED
2026-06-14 21:53:04,343 [WARNING] ID 17 INTERRUPTED
2026-06-14 21:53:04,372 [INFO] Completed in 0h 7m 30s
2026-06-14 21:53:04,372 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\17\data.tsv.gz


## HERA — recipe 2 gate map, RE-RUN at settled dwell (4τ = 1.2 s)

Supersedes `data/17` (run at 0.5 s = 1.67τ, under-settled — **interrupt cell 41 and discard it**). τ = 0.3 s, so dwell = **4τ = 1.2 s** per operator standard.

**Settled 2D megasweep**: slow = BG [−4.5,+4.5] × **17 rows** (0.5 V/row), fast = TG [−6,+7] × **41 pts** (0.325 V/pt), `fast_delay=1.2`, `slow_delay=2.0`. Reads the three 4T voltages + drain current. Trimmed from 21×53 to hold wall-clock against the 2.4× longer dwell; 17 rows still give a clean slope lever and 0.325 V/pt resolves the ~1 V CNP peak (smoothed argmax per recipe step 1).

**Price** ~19 min (17 rows × [41 × ~1.44 s/pt + 6.5 s TG reset + 2 s BG settle]); per-point from the 0.74 s/pt @ 0.5 s calibration → ~0.24 s overhead → ~1.44 s/pt @ 1.2 s. Writes `data/18`. Now that dwell is at 4τ, settling is adequate (no fwd/back lag check needed). **After**: re-form per channel, argmax ridge, polyfit slope → c_bg/c_tg, offset, D-line designer, write back; check sr2 usability across the plane.

In [29]:
# Recipe 2 gate map, settled dwell (4tc=1.2s). Re-selects blessed routing.
assert 1.0 < heater.sample_temperature() < 1.5, 'T off base'
assert abs(magnet.mag()) < 0.05, 'B not ~0'
assert abs(k_2410.volt()) < 0.01, 'Si not 0'

ROUTING_4T = {'source':'T2','drain':'T7',
              'sr1+':'T3','sr1-':'T4', 'sr2+':'B4','sr2-':'B5', 'sr3+':'T5','sr3-':'B6'}
mux.select(ROUTING_4T)                 # re-establish after interrupt; Si 16/11 untouched

BG_VEC = np.linspace(-4.5, 4.5, 17)    # slow, 0.5 V/row
TG_VEC = np.linspace(-6.0, 7.0, 41)    # fast, 0.325 V/pt

mc = {**meas_config,
      'comments': 'cnp-and-dline-calibration | 0T TG-BG gatemap SETTLED 4tc=1.2s | '
                  'blessed cfg src=T2 drn=T7 sr1=T3-T4 sr2=B4-B5 sr3=T5-B6 | '
                  'BG[-4.5,4.5]x17 TG[-6,7]x41 Si=0'}
mc = mc | mux.mc_fragment()
s = async_station(mc)
s.fp(sr0.X).fp(sr1.X).fp(sr2.X).fp(sr3.X).fp(sr0.Y)
s.add_comment('cnp-and-dline-calibration')
s.megasweep(k_bg.volt, BG_VEC, dc205_volt, TG_VEC, slow_delay=2.0, fast_delay=1.2)

2026-06-14 21:53:12,466 [INFO] Starting megasweep with ID 18
2026-06-14 21:53:12,466 [INFO] Minimum duration 0h 14m 30s


2026-06-14 22:12:30,311 [INFO] Completed in 0h 19m 17s
2026-06-14 22:12:30,311 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\18\data.tsv.gz


## HERA — Phase 2 step 3: D≈0 Landau fan (coarse first pass), B 0→8 T

**Recipe**: `landau-fan` + `trilayer-stacking-id` step 3. D = 0 (the clean regime for ABA crossing inference; finite-D presence is not ABA evidence). Electron side + CNP column: **n = +4 → −1 ×10¹² cm⁻² (71 pts)**, fast axis a paired (TG,BG) line at fixed D via the phase-1 calibration. **B = 0 → 8 T (81 rows, 0.1 T)** — magnet starts where it sits (0 T), ends at cap. Reads sr1/sr2 (Rxx) + sr3 (Rxy) + current + T.

**Why coarse**: device mobility is unknown, so this first pass finds the **onset field** (lowest B where LL structure resolves — a quality metric) and the **structure type** before a fine zoom. Grid resolves rays/SdH above ~2–3 T; **it aliases below ~2 T — I will not claim chain structure there** (recipe Nyquist guard), the zoom comes only on confirmed structure.

**Pre-fan setup** (separate cell): sr1/sr2 → 2 mV, sr3 → 1 mV, τ = 0.3 s on all — fixed, not autoranged (row-to-row comparability; high-B Rxx would rail the 0.5 mV range).

**Cost** ~3.5 h (81 rows × [0.1 T ramp ≈ 44 s + 71 × 1.44 s/pt measure ≈ 102 s + ~10 s retrace]); from the 1.44 s/pt (4τ) calibration + 0.137 T/min magnet. **Checkpoint** (background `hera-watch`): at ~row 40 (B ≈ 4 T, ~1.5 h) I read — channels not railing? structure appearing + onset B? T holding? → continue / re-range / note-resolution-limited. (A 3τ = 0.9 s dwell, recipe-sanctioned for position-reading, would cut this ~25% to ~2.7 h — say the word and I switch.) Writes `data/19`.

**After**: TSV-first re-form Rxx(n,B) per channel (step-2 triage, never averaged), ABA forward-model overlay (`hera/models/aba_trilayer`) for crossing loci, lever-arm check (ray dn/dB = ν·e/h), Středa.

In [30]:
# Pre-fan lock-in setup (recipe step 3) — fix sensitivity + TC BEFORE the fan.
for sr in (sr1, sr2, sr3):
    sr.time_constant(0.3)
sr1.sensitivity(2e-3); sr2.sensitivity(2e-3); sr3.sensitivity(1e-3)
print('sens sr1/sr2/sr3 =', sr1.sensitivity(), sr2.sensitivity(), sr3.sensitivity(), '| tc =', sr1.time_constant())

sens sr1/sr2/sr3 = 0.002 0.002 0.001 | tc = 0.3


In [31]:
# Phase 2 step 3 — D=0 Landau fan, electron side + CNP, B 0->8 T (coarse first pass).
assert 1.0 < heater.sample_temperature() < 1.5, 'T off base'
assert abs(k_2410.volt()) < 0.01, 'Si not 0'
assert abs(magnet.mag()) < 0.05, 'B not ~0 at start'

ROUTING_4T = {'source':'T2','drain':'T7','sr1+':'T3','sr1-':'T4','sr2+':'B4','sr2-':'B5','sr3+':'T5','sr3-':'B6'}
mux.select(ROUTING_4T)

from sharpelab_nb.gates import stack_capacitance, calc_gate_voltages
c_tg = stack_capacitance([(15.6,3)]); c_bg = 1.3136*c_tg; n_offset, D_offset = -0.225, 0.0
ns = np.linspace(4.0, -1.0, 71)                                   # electron side + CNP, D=0
Vtgs, Vbgs = calc_gate_voltages(ns, np.zeros_like(ns), c_tg, c_bg, n_offset, D_offset)
assert Vtgs.min() >= -6 and Vtgs.max() <= 7 and Vbgs.min() >= -4.5 and Vbgs.max() <= 4.5, 'n-line exits validators'
B_vec = np.linspace(0.0, 8.0, 81)                                 # slow, 0.1 T steps; starts at 0 T

mc = {**meas_config,
      'comments': 'landau-fan | D=0 V/nm n=4..-1 (1e12 cm^-2) x71 B=0..8 T x81 | coarse first pass | '
                  'cfg src=T2 drn=T7 sr1=T3-T4(Rxx) sr2=B4-B5(Rxx) sr3=T5-B6(Rxy)'}
mc = mc | mux.mc_fragment()
s = async_station(mc)
s.fp(sr0.X).fp(sr1.X).fp(sr2.X).fp(sr3.X).fp(sr3.Y).fp(heater.sample_temperature)
s.add_comment('landau-fan')
s.add_comment('D = 0 V/nm, n = 4 to -1, B = 0 to 8 T')
s.multimegasweep([magnet.mag], [B_vec], [dc205_volt, k_bg.volt], [Vtgs, Vbgs], slow_delay=2.0, fast_delay=1.2)

2026-06-14 22:34:52,311 [INFO] Starting multimegasweep with ID 19
2026-06-14 22:34:52,312 [INFO] Minimum duration 1h 57m 43s


VisaIOError: ('VI_ERROR_TMO (-1073807339): Timeout expired before operation completed.', "asking 'KRDG? A' to <Lakeshore_335: l335_3>", 'getting l335_3_sample_temperature')

## HERA — D≈0 fan RE-RUN (prior file_id 19 died on LS335 VISA contention)

**Why 19 died**: `s.fp(heater.sample_temperature)` polled the Lakeshore 335 every fast point → VISA contention with the Grafana logger (rig-card two-masters quirk) → `VI_ERROR_TMO` killed the run at row 2 (~7 min). My error — the rig card warns against per-point Lakeshore polling. file_id 19 is discarded (3 rows).

**Fix**: identical fan, **T removed from the per-point follow-list** (base T is logged externally by Grafana; not needed in the TSV). Single preflight T read kept. Sensitivities/τ already set (cell 47). Magnet sits at ~0.2–0.3 T from the aborted run — the fan's B_vec[0]=0 ramps it back to 0 first (~2 min), then ascends. **Same plan**: D=0, n=+4→−1 ×71, B 0→8 T ×81, ~3.5 h, writes `data/20`. **Checkpoint** at ~row 40 via `hera-watch` (re-armed when you start it).

In [32]:
# Phase 2 step 3 — D=0 Landau fan RE-RUN. T removed from fp (LS335 VISA contention killed file_id 19).
assert 1.0 < heater.sample_temperature() < 1.5, 'T off base'   # single read OK; per-point T poll removed
assert abs(k_2410.volt()) < 0.01, 'Si not 0'
assert abs(magnet.mag()) < 1.0, 'B unexpectedly high'          # ~0.2-0.3 T from aborted run; fan ramps to 0
print('B at start =', magnet.mag(), 'T (fan ramps to 0 first)')

ROUTING_4T = {'source':'T2','drain':'T7','sr1+':'T3','sr1-':'T4','sr2+':'B4','sr2-':'B5','sr3+':'T5','sr3-':'B6'}
mux.select(ROUTING_4T)

from sharpelab_nb.gates import stack_capacitance, calc_gate_voltages
c_tg = stack_capacitance([(15.6,3)]); c_bg = 1.3136*c_tg; n_offset, D_offset = -0.225, 0.0
ns = np.linspace(4.0, -1.0, 71)
Vtgs, Vbgs = calc_gate_voltages(ns, np.zeros_like(ns), c_tg, c_bg, n_offset, D_offset)
assert Vtgs.min() >= -6 and Vtgs.max() <= 7 and Vbgs.min() >= -4.5 and Vbgs.max() <= 4.5, 'n-line exits validators'
B_vec = np.linspace(0.0, 8.0, 81)

mc = {**meas_config,
      'comments': 'landau-fan | D=0 V/nm n=4..-1 (1e12 cm^-2) x71 B=0..8 T x81 | coarse first pass (rerun of 19) | '
                  'cfg src=T2 drn=T7 sr1=T3-T4(Rxx) sr2=B4-B5(Rxx) sr3=T5-B6(Rxy) | T not fp (LS335 contention)'}
mc = mc | mux.mc_fragment()
s = async_station(mc)
s.fp(sr0.X).fp(sr1.X).fp(sr2.X).fp(sr3.X).fp(sr3.Y)   # NO sample_temperature fp (LS335 two-masters)
s.add_comment('landau-fan')
s.add_comment('D = 0 V/nm, n = 4 to -1, B = 0 to 8 T')
s.multimegasweep([magnet.mag], [B_vec], [dc205_volt, k_bg.volt], [Vtgs, Vbgs], slow_delay=2.0, fast_delay=1.2)

B at start = 0.20235964383561644 T (fan ramps to 0 first)
2026-06-14 22:49:19,135 [INFO] Starting multimegasweep with ID 20
2026-06-14 22:49:19,135 [INFO] Minimum duration 1h 57m 43s


2026-06-15 02:10:54,733 [INFO] Completed in 3h 21m 35s
2026-06-15 02:10:54,733 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\20\data.tsv.gz


## HERA — Phase 2 firming fan: high-n D=0, ride magnet DOWN 8 to 0 T

**Why**: the D=0 stacking call is ABC (weak-to-moderate); the one resolvable gap is that the ABA crossing-chain region (N_M=1 at n~6.8, N_M=2 at n~10) sits at **n>4, outside file_id 20's window**. This fan covers n=3.5 to 10 at D=0 (the D=0 line reaches n~12 inside the validator box) — chain-absence there firms ABC; chains appearing would force a re-think. Also extends the single-pocket SdH-FFT test to higher n.

**Efficiency**: magnet is at 8 T from file_id 20; this fan is **B descending 8 to 0** — it starts where the magnet sits and rides it home, no re-traverse. If you are NOT continuing tonight, this is also the productive way to bring B back to 0; otherwise ramp the magnet to 0 manually.

**Design**: D=0, n=10 to 3.5 (71 pts, clipped to validators), B 8 to 0 T (81 rows, 0.1 T). T NOT in the per-point follow-list (LS335 contention — same fix as file_id 20). Sensitivities already at 2/2/1 mV from cell 47 (Rxx is small at high n, fine). ~3.4 h, writes data/21. **Checkpoint** at ~row 40 (B~4 T) via hera-watch when you start it. Same per-point cost as file_id 20.

In [ ]:
# Phase 2 firming fan: high-n D=0, B descending 8->0 (rides magnet home). NO T fp (LS335).
assert 1.0 < heater.sample_temperature() < 1.5, 'T off base'
assert abs(k_2410.volt()) < 0.01, 'Si not 0'
print('B at start =', magnet.mag(), 'T (expect ~8; fan descends to 0)')

ROUTING_4T = {'source':'T2','drain':'T7','sr1+':'T3','sr1-':'T4','sr2+':'B4','sr2-':'B5','sr3+':'T5','sr3-':'B6'}
mux.select(ROUTING_4T)

from sharpelab_nb.gates import stack_capacitance, calc_gate_voltages
c_tg = stack_capacitance([(15.6,3)]); c_bg = 1.3136*c_tg; n_offset, D_offset = -0.225, 0.0
ns_req = np.linspace(10.0, 3.5, 71)
Vtgs, Vbgs = calc_gate_voltages(ns_req, np.zeros_like(ns_req), c_tg, c_bg, n_offset, D_offset)
keep = (Vtgs >= -6) & (Vtgs <= 7) & (Vbgs >= -4.5) & (Vbgs <= 4.5)
ns, Vtgs, Vbgs = ns_req[keep], Vtgs[keep], Vbgs[keep]
print(f'n-line realized: {ns.min():.2f}..{ns.max():.2f} ({keep.sum()}/{len(ns_req)} pts in box)')
B_vec = np.linspace(8.0, 0.0, 81)

mc = {**meas_config,
      'comments': 'landau-fan | D=0 V/nm high-n n=10..3.5 x'+str(int(keep.sum()))+' B=8..0 T x81 | firming fan for ABA chain region | '
                  'cfg src=T2 drn=T7 sr1=T3-T4 sr2=B4-B5 sr3=T5-B6 | T not fp (LS335)'}
mc = mc | mux.mc_fragment()
s = async_station(mc)
s.fp(sr0.X).fp(sr1.X).fp(sr2.X).fp(sr3.X).fp(sr3.Y)
s.add_comment('landau-fan')
s.add_comment('D = 0 V/nm, n = 10 to 3.5, B = 8 to 0 T (high-n firming)')
s.multimegasweep([magnet.mag], [B_vec], [dc205_volt, k_bg.volt], [Vtgs, Vbgs], slow_delay=2.0, fast_delay=1.2)

B at start = 8.002352260273973 T (expect ~8; fan descends to 0)
n-line realized: 3.50..10.00 (71/71 pts in box)
2026-06-15 09:07:16,526 [INFO] Starting multimegasweep with ID 21
2026-06-15 09:07:16,527 [INFO] Minimum duration 1h 57m 43s


## HERA — multi-region D-gap survey (B=0): whole-device ABC vs mosaic

Shores up the ABC call without temperature. A displacement-field gap at the CNP is the ABC fingerprint (ABA has none), so extracting CNP-Rxx-vs-D across more of the device tests whether the gap is device-wide (uniform ABC) or only on the two paths measured so far (the step-1 mosaic caveat). **Magnet stays at 0 T.**

Two new mux routings (source T2 / drain T7 fixed), each a 0 T TG-BG gate map under the **same settled conditions as file_id 18** (BG +/-4.5 x17, TG -6..7 x41, 4tau dwell), so directly comparable:
- **Routing B**: sr1=T4-T5, sr2=B3-B4, sr3=B6-B7
- **Routing C**: sr1=T3-T5, sr2=B4-B6, sr3=B3-B5

With file_id 18 (T3-T4, B4-B5) this covers 8 longitudinal regions across both edges. ~17 min each, ~35 min total; writes data/22-23. **T NOT in the per-point follow list** (LS335 contention). Sensitivities carried from the fan (2/2/1 mV; R is sensitivity-independent, so comparable to file_id 18). **After**: I overlay CNP-Rxx-vs-D for all regions — all gapping = device-wide ABC; any flat region = genuine mixed-domain finding. Preflight asserts T base, B~0, Si=0.

In [34]:
# Multi-region D-gap survey: 0T TG-BG gate maps on 2 new probe-pair routings. Tests whole-device ABC vs mosaic.
assert 1.0 < heater.sample_temperature() < 1.5, 'T off base'
assert abs(magnet.mag()) < 0.05, 'B not ~0 (survey is B=0)'
assert abs(k_2410.volt()) < 0.01, 'Si not 0'

BG_VEC = np.linspace(-4.5, 4.5, 17)   # slow, matches file_id 18
TG_VEC = np.linspace(-6.0, 7.0, 41)   # fast, matches file_id 18
ROUTINGS = {
    'B': {'source':'T2','drain':'T7','sr1+':'T4','sr1-':'T5','sr2+':'B3','sr2-':'B4','sr3+':'B6','sr3-':'B7'},
    'C': {'source':'T2','drain':'T7','sr1+':'T3','sr1-':'T5','sr2+':'B4','sr2-':'B6','sr3+':'B3','sr3-':'B5'},
}
for tag, rt in ROUTINGS.items():
    mux.select(rt)                                  # B=0; Si 16/11 untouched
    frag = mux.mc_fragment()
    mc = {**meas_config,
          'comments': f'cnp-and-dline-calibration | D-gap multi-region survey routing {tag} | 0T TG-BG gatemap B=0 | '
                      f"sr1={frag.get('sr1')} sr2={frag.get('sr2')} sr3={frag.get('sr3')}"}
    mc = mc | frag
    s = async_station(mc)
    s.fp(sr0.X).fp(sr1.X).fp(sr2.X).fp(sr3.X)       # NO sample_temperature fp (LS335 two-masters)
    s.add_comment('cnp-and-dline-calibration')
    s.add_comment(f'D-gap survey routing {tag}, B=0')
    s.megasweep(k_bg.volt, BG_VEC, dc205_volt, TG_VEC, slow_delay=2.0, fast_delay=1.2)
    print('done routing', tag)

2026-06-15 15:13:46,739 [INFO] Starting megasweep with ID 22
2026-06-15 15:13:46,740 [INFO] Minimum duration 0h 14m 30s


2026-06-15 15:32:56,448 [INFO] Completed in 0h 19m 9s
2026-06-15 15:32:56,449 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\22\data.tsv.gz
done routing B
2026-06-15 15:33:01,073 [INFO] Starting megasweep with ID 23
2026-06-15 15:33:01,074 [INFO] Minimum duration 0h 14m 30s


2026-06-15 15:52:16,733 [INFO] Completed in 0h 19m 15s
2026-06-15 15:52:16,734 [INFO] Data saved in C:\Users\DGG\VTI_local_data\zack\20260614-QC0426\data\23\data.tsv.gz
done routing C


## HERA — session park (clean idle state)

End of session. Ramp gates to 0, ground the device contacts via the mux (Si lines 16/11 are NOT in the managed set, so they stay closed — Si gate holds). Magnet is already at 0 T from the last fan; base T holds on the closed-loop cooler. Run this to leave the rig parked; skip it only if you're keeping the device biased for an immediate continuation.

In [ ]:
# Session park.
dc205_volt(0.0)        # TG -> 0 (DC205 ramp 2 V/s)
k_bg.volt(0.0)         # BG -> 0 (0.005 V steps)
mux.park()             # ground device contacts; Si lines 16/11 not managed -> stay closed
print('parked: TG=', k_tg.voltage(), 'BG=', k_bg.volt(), 'Si=', k_2410.volt(), 'B=', magnet.mag())
print('qsw state:', qsw.state())